<a href="https://colab.research.google.com/github/GOWTHAMVANTAKULA/Financial-Forecasting-Frontier-Distributed-ML/blob/main/Predictive_Modeling_for_Banking_Trends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1.  Data Loading and Initial Exploration:**

**1.  Load the bank.csv dataset into a Spark DataFrame.**

In [ ]:
# Step 1: Setup PySpark in Google Colab

# Install PySpark
!pip install -q pyspark==3.5.1

# Import SparkSession
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BankMarketingAnalysis") \
    .getOrCreate()

print("✅ Spark Session Created Successfully")
print("Spark Version:", spark.version)

✅ Spark Session Created Successfully
Spark Version: 3.5.1


In [ ]:
# Load bank.csv

df = spark.read.csv(
    "/content/drive/MyDrive/Financial Forecasting Frontier: Distributed ML/bank.csv",
    header=True,
    inferSchema=True
)

print("✅ Dataset Loaded Successfully")

✅ Dataset Loaded Successfully


**2. Perform basic exploratory data analysis (EDA) to understand the dataset. Display the schema and the first few rows.**

In [ ]:
# Display first 5 rows
print("First 5 Rows:")
df.show(5, truncate=False)

# Display schema
print("\nDataset Schema:")
df.printSchema()

# Display number of rows and columns
print("\nDataset Dimensions:")
print("Rows:", df.count())
print("Columns:", len(df.columns))

# Display column names
print("\nColumn Names:")
print(df.columns)

# Display basic summary statistics
print("\nSummary Statistics:")
df.describe().show()

First 5 Rows:
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|job        |marital|education|default|balance|housing|loan|contact |day|month|duration|campaign|pdays|previous|poutcome|y  |
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|30 |unemployed |married|primary  |no     |1787   |no     |no  |cellular|19 |oct  |79      |1       |-1   |0       |unknown |no |
|33 |services   |married|secondary|no     |4789   |yes    |yes |cellular|11 |may  |220     |1       |339  |4       |failure |no |
|35 |management |single |tertiary |no     |1350   |yes    |no  |cellular|16 |apr  |185     |1       |330  |1       |failure |no |
|30 |management |married|tertiary |no     |1476   |yes    |yes |unknown |3  |jun  |199     |4       |-1   |0       |unknown |no |
|59 |blue-collar|married|secondary|no     |0      |yes    |no  |unknown |5  

**2. Data Preprocessing:**

* Handle missing values in the dataset.
* Handle Outliers in the dataset
* Convert categorical variables to numerical format using StringIndexer or OneHotEncoder.

In [ ]:
from pyspark.sql.functions import col, when
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# 1. Handle Missing Values
print("Missing Values Before Handling:")

for column in df.columns:
    missing_count = df.filter(
        col(column).isNull() | (col(column) == "")
    ).count()
    print(f"{column}: {missing_count}")

# Replace empty strings with null
df_clean = df

for column in df_clean.columns:
    df_clean = df_clean.withColumn(
        column,
        when(col(column) == "", None).otherwise(col(column))
    )

# Fill missing numerical columns with median
numerical_cols = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

for column in numerical_cols:
    median_value = df_clean.approxQuantile(
        column, [0.5], 0.01
    )[0]

    df_clean = df_clean.fillna(
        {column: median_value}
    )

# Fill missing categorical columns with "unknown"
categorical_cols = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome"
]

df_clean = df_clean.fillna(
    {column: "unknown" for column in categorical_cols}
)

# Fill missing target values
df_clean = df_clean.dropna(subset=["y"])

print("\n✅ Missing values handled successfully")

# 2. Handle Outliers
print("\nHandling Outliers...")

# Calculate IQR and cap outliers
for column in numerical_cols:

    quantiles = df_clean.approxQuantile(
        column, [0.25, 0.75], 0.01
    )

    Q1 = quantiles[0]
    Q3 = quantiles[1]

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap values instead of removing records
    df_clean = df_clean.withColumn(
        column,
        when(col(column) < lower_bound, lower_bound)
        .when(col(column) > upper_bound, upper_bound)
        .otherwise(col(column))
    )

print("✅ Outliers handled using IQR-based capping")

# 3. Convert Categorical Variables
# Convert categorical columns into numerical indexes
indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=column + "_index",
        handleInvalid="keep"
    )
    for column in categorical_cols
]

# Apply StringIndexer
for indexer in indexers:
    df_clean = indexer.fit(df_clean).transform(df_clean)

# One-hot encode categorical variables
encoder = OneHotEncoder(
    inputCols=[column + "_index" for column in categorical_cols],
    outputCols=[column + "_encoded" for column in categorical_cols]
)

encoder_model = encoder.fit(df_clean)

df_clean = encoder_model.transform(df_clean)

print("✅ Categorical variables converted using StringIndexer and OneHotEncoder")

# Final Dataset
print("\nFinal Preprocessed Dataset:")
df_clean.show(5, truncate=False)

print("\nFinal Schema:")
df_clean.printSchema()

Missing Values Before Handling:
age: 0
job: 0
marital: 0
education: 0
default: 0
balance: 0
housing: 0
loan: 0
contact: 0
day: 0
month: 0
duration: 0
campaign: 0
pdays: 0
previous: 0
poutcome: 0
y: 0

✅ Missing values handled successfully

Handling Outliers...
✅ Outliers handled using IQR-based capping
✅ Categorical variables converted using StringIndexer and OneHotEncoder

Final Preprocessed Dataset:
+----+-----------+-------+---------+-------+-------+-------+----+--------+----+-----+--------+--------+-----+--------+--------+---+---------+-------------+---------------+-------------+-------------+----------+-------------+-----------+--------------+--------------+---------------+-----------------+---------------+---------------+-------------+---------------+--------------+----------------+
|age |job        |marital|education|default|balance|housing|loan|contact |day |month|duration|campaign|pdays|previous|poutcome|y  |job_index|marital_index|education_index|default_index|housing_index|l

**3. Feature Engineering and Data Transformation:**

**Create a feature vector using VectorAssembler by combining all feature columns.**

In [ ]:
# FEATURE ENGINEERING
from pyspark.ml.feature import VectorAssembler

# Numerical feature columns
numerical_cols = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

# One-hot encoded categorical columns
encoded_cols = [
    "job_encoded",
    "marital_encoded",
    "education_encoded",
    "default_encoded",
    "housing_encoded",
    "loan_encoded",
    "contact_encoded",
    "month_encoded",
    "poutcome_encoded"
]

# Combine all feature columns
feature_cols = numerical_cols + encoded_cols

# Create VectorAssembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Create feature vector
df_features = assembler.transform(df_clean)

# Display the feature vector
print("Feature Vector:")
df_features.select(
    "age",
    "job",
    "balance",
    "y",
    "features"
).show(5, truncate=False)

# Display the final schema
print("\nFinal Schema:")
df_features.printSchema()

print("\n✅ Feature vector created successfully!")

Feature Vector:
+----+-----------+-------+---+-------------------------------------------------------------------------------------------------------------------+
|age |job        |balance|y  |features                                                                                                           |
+----+-----------+-------+---+-------------------------------------------------------------------------------------------------------------------+
|30.0|unemployed |1787.0 |no |(51,[0,1,2,3,4,5,15,19,24,26,29,30,32,43,47],[30.0,1787.0,19.0,79.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |
|33.0|services   |3461.0 |no |(51,[0,1,2,3,4,5,11,19,22,26,28,31,32,35,48],[33.0,3461.0,11.0,220.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|35.0|management |1350.0 |no |(51,[0,1,2,3,4,5,7,20,23,26,28,30,32,40,48],[35.0,1350.0,16.0,185.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |
|30.0|management |1476.0 |no |(51,[0,1,2,3,4,5,7,19,23,26,28,31,33,38,47],[30.0,1476.0,3.0,199.0,4.0,-

### **4. Model Training and Selection:**

* Choose a classification model (e.g., Logistic Regression, Decision Tree Classifier) for predicting the subscription to a term deposit.
* Split the data into training and test sets.
* Train the model on the training dataset.


In [ ]:

# MODEL TRAINING - LOGISTIC REGRESSION
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression

# 1. Convert Target Variable (y) to Label
label_indexer = StringIndexer(
    inputCol="y",
    outputCol="label"
)

label_model = label_indexer.fit(df_features)

df_labeled = label_model.transform(df_features)

# 2. Select Required Columns
model_data = df_labeled.select(
    "features",
    "label",
    "y"
)

# 3. Split Dataset into Training and Testing
train_data, test_data = model_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training Records:", train_data.count())
print("Testing Records:", test_data.count())

# 4. Create Logistic Regression Model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100
)

# 5. Train the Model
lr_model = lr.fit(train_data)

print("\n✅ Logistic Regression Model Trained Successfully!")

# 6. Make Predictions
predictions = lr_model.transform(test_data)

print("\nSample Predictions:")
predictions.select(
    "y",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

Training Records: 3662
Testing Records: 859

✅ Logistic Regression Model Trained Successfully!

Sample Predictions:
+---+-----+----------+-----------------------------------------+
|y  |label|prediction|probability                              |
+---+-----+----------+-----------------------------------------+
|yes|1.0  |1.0       |[0.4919323631985812,0.5080676368014188]  |
|no |0.0  |0.0       |[0.8865855968621081,0.11341440313789186] |
|no |0.0  |0.0       |[0.9751603778145914,0.024839622185408583]|
|no |0.0  |0.0       |[0.9779255999345301,0.0220744000654699]  |
|no |0.0  |0.0       |[0.8092161592371664,0.1907838407628336]  |
|no |0.0  |0.0       |[0.9872844316719736,0.01271556832802645] |
|no |0.0  |0.0       |[0.9816103411922539,0.018389658807746123]|
|no |0.0  |0.0       |[0.7652681050746206,0.23473189492537938] |
|no |0.0  |0.0       |[0.9273612858107391,0.07263871418926093] |
|no |0.0  |0.0       |[0.9127997351651514,0.08720026483484855] |
+---+-----+----------+-----------------

**5. Model Evaluation:**

* Evaluate the model on the test dataset using appropriate metrics.

In [ ]:
# STEP 5: MODEL EVALUATION
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

# 2. Precision
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

precision = precision_evaluator.evaluate(predictions)

# 3. Recall
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

recall = recall_evaluator.evaluate(predictions)

# 4. F1 Score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

f1_score = f1_evaluator.evaluate(predictions)

print("📊 Logistic Regression Model Evaluation")
print("----------------------------------------")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1_score:.4f}")

📊 Logistic Regression Model Evaluation
----------------------------------------
Accuracy  : 0.8964
Precision : 0.8812
Recall    : 0.8964
F1 Score  : 0.8836


**6. Hyperparameter Tuning:**

* Perform hyperparameter tuning (using ParamGridBuilder and CrossValidator.)

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .addGrid(lr.maxIter, [50, 100]) \
    .build()

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

cross_validator = CrossValidator(
    estimator=lr,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cv_model = cross_validator.fit(train_data)

best_lr_model = cv_model.bestModel

print("Best Logistic Regression Model:")
print("Regularization Parameter:", best_lr_model.getRegParam())
print("Elastic Net Parameter:", best_lr_model.getElasticNetParam())
print("Maximum Iterations:", best_lr_model.getMaxIter())

tuned_predictions = best_lr_model.transform(test_data)

auc = evaluator.evaluate(tuned_predictions)

print("\nTuned Model AUC:", round(auc, 4))
print("✅ Hyperparameter tuning completed successfully")

Best Logistic Regression Model:
Regularization Parameter: 0.1
Elastic Net Parameter: 0.0
Maximum Iterations: 50

Tuned Model AUC: 0.8947
✅ Hyperparameter tuning completed successfully


**7. Advanced Analysis:**

* Analyze the feature importances (if applicable) or coefficients of the model to gain insights into which features are most influential in predicting the outcome.



In [ ]:
from pyspark.sql.functions import col

coefficients = best_lr_model.coefficients.toArray()

coefficient_data = [
    (feature, float(coefficient))
    for feature, coefficient in zip(feature_names, coefficients)
]

coefficient_df = spark.createDataFrame(
    coefficient_data,
    ["feature", "coefficient"]
)

print("Features with Positive Coefficients:")

coefficient_df.orderBy(
    col("coefficient").desc()
).show(15, truncate=False)

print("Features with Negative Coefficients:")

coefficient_df.orderBy(
    col("coefficient").asc()
).show(15, truncate=False)

Features with Positive Coefficients:
+-----------------+---------------------+
|feature          |coefficient          |
+-----------------+---------------------+
|loan_encoded     |0.3841532305062732   |
|job_encoded      |0.057320582876609655 |
|contact_encoded  |0.04503352687571682  |
|default_encoded  |0.041169377765458884 |
|duration         |0.003132820560513465 |
|day              |0.0022267860230886253|
|age              |0.0019608114378226633|
|balance          |5.4056204180146754E-5|
|pdays            |0.0                  |
|previous         |0.0                  |
|campaign         |-0.03986691263096992 |
|education_encoded|-0.0473814875272139  |
|housing_encoded  |-0.07885747613931517 |
|month_encoded    |-0.09226630946574312 |
|marital_encoded  |-0.11845526821141619 |
+-----------------+---------------------+
only showing top 15 rows

Features with Negative Coefficients:
+-----------------+---------------------+
|feature          |coefficient          |
+-----------------

**Conclusion**

* The goal of the project is to predict whether a bank customer will subscribe to a term deposit or not using the bank.csv dataset.

* A Logistic Regression binary classification model was built using Spark ML, where:

          yes → Customer subscribes to the term deposit
          no → Customer does not subscribe

* The model uses customer information such as age, job, balance, housing loan, personal loan, contact method, campaign details, and previous campaign outcomes to make the prediction.

* The dataset was preprocessed, transformed into feature vectors, and divided into training and testing datasets. The Logistic Regression model was then trained and evaluated using Accuracy, Precision, Recall, F1-score, and AUC-ROC.

* Therefore, the Spark ML model helps predict the likelihood of a customer subscribing to a term deposit based on their available banking and campaign information.